# Lab 4.2 – Kundenfeedback auswerten

**Beispiel.** Ausgeführt ohne API-Schlüssel: Alle Einordnungen stammen aus den **Stichwortregeln** (Spalte `methode`), nicht vom Sprachmodell.
Zellen der Reihe nach mit **Umschalt+Enter** ausführen.

| Schritt | Ergebnis |
| --- | --- |
| 50 Kommentare mit `classify()` einordnen | Spalten `stimmung`, `thema`, `methode` |
| Trefferquote an den 20 von Hand gelabelten messen | `treffer_thema`, `treffer_stimmung`  |
| Themen zählen und als Balken zeigen | `diagramm_themen.html` |
| Ergebnis speichern | `feedback_klassifiziert.csv` |

In [1]:
import json
import os

import pandas as pd
import plotly.express as px

MODELL = "gpt-4.1-mini"                  # derselbe Kurs-Schlüssel wie in Lab 4.1
EINGABE = "data/nordwind_feedback.csv"
ORDNER = "project/examples/m04-llm-analysis/l02-feedback/"
MIT_DIENST = bool(os.environ.get("OPENAI_API_KEY"))

feedback = pd.read_csv(EINGABE)
kommentare = feedback.head(50).copy()    # die ersten 50 Kommentare
print(len(feedback), "Kommentare; davon in den ersten 50 von Hand gelabelt:", kommentare["thema_label"].notna().sum())
print("Sprachmodell:" if MIT_DIENST else "Ohne Dienst (OPENAI_API_KEY fehlt): classify() nutzt Stichwortregeln.", MODELL if MIT_DIENST else "")
kommentare.head()

300 Kommentare; davon in den ersten 50 von Hand gelabelt: 20
Ohne Dienst (OPENAI_API_KEY fehlt): classify() nutzt Stichwortregeln. 


,id,datum,kanal,kommentar,thema_label,stimmung_label
0,1,2026-07-24,Außendienst,Bestellung im Shop ging schnell.,Onlineshop,positiv
1,2,2026-09-30,Onlineshop,Preise sind fair für die Qualität.,NaN,NaN
2,3,2026-04-20,Onlineshop,Bei der Konkurrenz ist Toner billiger.,NaN,NaN
3,4,2026-07-09,Onlineshop,Kugelschreiber schreiben nach einer Woche nich...,NaN,NaN
4,5,2026-06-08,Onlineshop,Gibt es einen neuen Ansprechpartner?,Service,neutral


## Der Prompt

Der Text in `PROMPT` geht zusammen mit dem Kommentar an das Modell. Feste Felder und feste Werte machen die Antwort auswertbar.

In [2]:
PROMPT = """Ordne einen Kundenkommentar eines Großhändlers für Büro- und Haushaltsbedarf ein.
Antworte nur mit JSON in dieser Form: {"stimmung": "...", "thema": "..."}
stimmung: genau einer der Werte positiv, neutral, negativ
thema: genau einer der Werte Lieferung, Preis, Qualität, Onlineshop, Service
Kommentar: """

## `classify()` (fertig)

Mit Schlüssel fragt `classify()` das Sprachmodell. Ohne Schlüssel oder bei einer Störung ordnen **Stichwortregeln** ein: feste Wortlisten, kein Sprachmodell. Die Spalte `methode` zeigt bei jeder Zeile, was eingeordnet hat.

In [3]:
# Ersatz ohne Dienst: Stichwortregeln (kein Sprachmodell). Erste passende Themenliste gewinnt.
THEMA_STICHWORTE = {
    "Lieferung": ["liefer", "versand", "paket", "karton"],
    "Preis": ["preis", "billig", "teuer", "angebot"],
    "Qualität": ["qualität", "staut", "verarbeitet", "wirkt", "kaputt"],
    "Onlineshop": ["shop", "login", "warenkorb", "klick"],
    "Service": ["service", "hotline", "ansprechpartner", "ruft", "rechnung", "außendienst"],
}
NEGATIV = ["nicht", "spät", "verspätung", "besetzt", "niemand", "fehlen", "unvollständig",
           "gelöscht", "ärgerlich", "nachgelassen", "gestiegen", "billiger", "staut"]
POSITIV = ["schnell", "gut", "super", "danke", "toll", "zuverlässig", "fair", "übersichtlich",
           "pünktlich", "stabil", "klappt jetzt"]


def stichwortregeln(text):
    t = text.lower()
    thema = next((name for name, woerter in THEMA_STICHWORTE.items() if any(w in t for w in woerter)), "unklar")
    if any(w in t for w in NEGATIV):
        stimmung = "negativ"
    elif any(w in t for w in POSITIV):
        stimmung = "positiv"
    else:
        stimmung = "neutral"
    return {"stimmung": stimmung, "thema": thema, "methode": "Stichwortregeln"}


def classify(text, prompt=PROMPT):
    """Ordnet einen Kommentar ein: stimmung, thema und methode (Sprachmodell oder Stichwortregeln)."""
    if MIT_DIENST:
        try:
            import litellm
            antwort = litellm.completion(
                model=MODELL,
                messages=[{"role": "user", "content": prompt + text}],
                temperature=0,                              # möglichst gleiche Antwort bei gleicher Frage
                response_format={"type": "json_object"},   # Antwort muss JSON sein
            )
            felder = json.loads(antwort.choices[0].message.content)
            return {"stimmung": felder.get("stimmung"), "thema": felder.get("thema"), "methode": "Sprachmodell"}
        except Exception as fehler:                         # Netz, Schlüssel, Kontingent, kein JSON …
            print("Sprachmodell nicht erreichbar:", type(fehler).__name__, "– Stichwortregeln für:", text)
    return stichwortregeln(text)


classify("Lieferung kam drei Tage zu spät.")

{'stimmung': 'negativ', 'thema': 'Lieferung', 'methode': 'Stichwortregeln'}

### Grenze der Stichwortregeln

Die Wortlisten passen auf die typischen Formulierungen dieser Kommentare. Neue Formulierungen treffen sie oft nicht.

In [4]:
for text in ["Toner kam zerbrochen an.", "Rechnung stimmt schon wieder nicht.", "Lieferung kam, endlich."]:
    print(f"{text:<38} ->", stichwortregeln(text))

Toner kam zerbrochen an.               -> {'stimmung': 'neutral', 'thema': 'unklar', 'methode': 'Stichwortregeln'}
Rechnung stimmt schon wieder nicht.    -> {'stimmung': 'negativ', 'thema': 'Service', 'methode': 'Stichwortregeln'}
Lieferung kam, endlich.                -> {'stimmung': 'neutral', 'thema': 'Lieferung', 'methode': 'Stichwortregeln'}


## 50 Kommentare einordnen

Mit Sprachmodell: 50 Anfragen, je Anfrage ein Kommentar.

In [5]:
ergebnisse = [classify(text) for text in kommentare["kommentar"]]
kommentare = kommentare.join(pd.DataFrame(ergebnisse, index=kommentare.index))
print(kommentare["methode"].value_counts().to_string())
kommentare[["id", "kommentar", "stimmung", "thema", "methode"]].head(10)

methode
Stichwortregeln    50


,id,kommentar,stimmung,thema,methode
0,1,Bestellung im Shop ging schnell.,positiv,Onlineshop,Stichwortregeln
1,2,Preise sind fair für die Qualität.,positiv,Preis,Stichwortregeln
2,3,Bei der Konkurrenz ist Toner billiger.,negativ,Preis,Stichwortregeln
3,4,Kugelschreiber schreiben nach einer Woche nich...,negativ,unklar,Stichwortregeln
4,5,Gibt es einen neuen Ansprechpartner?,neutral,Service,Stichwortregeln
5,6,Papier staut ständig im Drucker.,negativ,Qualität,Stichwortregeln
6,7,Preise sind fair für die Qualität.,positiv,Preis,Stichwortregeln
7,8,Bitte Rechnung künftig als PDF schicken.,neutral,Service,Stichwortregeln
8,9,Hotline ist dauernd besetzt.,negativ,Service,Stichwortregeln
9,10,Niemand ruft zurück.,negativ,Service,Stichwortregeln


## Trefferquote an den 20 gelabelten Kommentaren

Trefferquote = Anteil der Kommentare, bei denen die Einordnung mit dem Label von Hand übereinstimmt.

In [6]:
gelabelt = kommentare[kommentare["thema_label"].notna()]     # nur die 20 von Hand gelabelten
gelabelt[["kommentar", "thema_label", "thema", "stimmung_label", "stimmung"]]

,kommentar,thema_label,thema,stimmung_label,stimmung
0,Bestellung im Shop ging schnell.,Onlineshop,Onlineshop,positiv,positiv
4,Gibt es einen neuen Ansprechpartner?,Service,Service,neutral,neutral
5,Papier staut ständig im Drucker.,Qualität,Qualität,negativ,negativ
7,Bitte Rechnung künftig als PDF schicken.,Service,Service,neutral,neutral
8,Hotline ist dauernd besetzt.,Service,Service,negativ,negativ
12,Schon wieder Verspätung beim Versand.,Lieferung,Lieferung,negativ,negativ
14,Kugelschreiber schreiben nach einer Woche nich...,Qualität,unklar,negativ,negativ
20,Warenkorb wurde beim Bezahlen gelöscht.,Onlineshop,Onlineshop,negativ,negativ
21,Schnelle Antwort vom Kundenservice.,Service,Service,positiv,positiv
23,Bestellung im Shop ging schnell.,Onlineshop,Onlineshop,positiv,positiv


In [7]:
# >>> AUFGABE 1: Trefferquoten treffer_thema und treffer_stimmung berechnen – Prompt siehe Übung, Aufgabe 3
treffer_thema = (gelabelt["thema"] == gelabelt["thema_label"]).mean()          # Anteil gleicher Zeilen
treffer_stimmung = (gelabelt["stimmung"] == gelabelt["stimmung_label"]).mean()
print(f"Trefferquote Thema:    {treffer_thema:.0%}")
print(f"Trefferquote Stimmung: {treffer_stimmung:.0%}")
# <<< ENDE AUFGABE 1

Trefferquote Thema:    95%
Trefferquote Stimmung: 100%


In [8]:
# Prüfzelle – nicht ändern
def pruefe_quote(name, wert, spalte, label):
    gleich = sum(a == b for a, b in zip(gelabelt[spalte], gelabelt[label]))
    if not isinstance(wert, (int, float)):
        return f"nicht OK – {name} ist noch keine Zahl."
    wert = wert / 100 if wert > 1 else wert                   # auch in Prozent erlaubt
    if abs(wert - gleich / len(gelabelt)) > 0.001:
        return f"nicht OK – {name} = {wert:.2f}: Anteil gleicher Werte in {spalte} und {label}, nur gelabelte Zeilen."
    return f"OK – {name}: {gleich} von {len(gelabelt)} gleich ({wert:.0%})"


print(pruefe_quote("treffer_thema", treffer_thema, "thema", "thema_label"))
print(pruefe_quote("treffer_stimmung", treffer_stimmung, "stimmung", "stimmung_label"))

OK – treffer_thema: 19 von 20 gleich (95%)
OK – treffer_stimmung: 20 von 20 gleich (100%)


## Themen zählen und zeigen

In [9]:
themen = kommentare["thema"].value_counts().rename_axis("thema").reset_index(name="anzahl")
methode = "Stichwortregeln" if (kommentare["methode"] == "Stichwortregeln").any() else "Sprachmodell"
fig = px.bar(
    themen, x="thema", y="anzahl", text="anzahl",
    title=f"{themen['thema'].iloc[0]} ist das häufigste Thema in den ersten 50 Kommentaren (eingeordnet per {methode})",
    labels={"thema": "Thema", "anzahl": "Kommentare"},
)
fig.write_html(ORDNER + "diagramm_themen.html", include_plotlyjs="cdn")
fig

## Ergebnis speichern

In [10]:
spalten = ["id", "datum", "kanal", "kommentar", "stimmung", "thema", "methode", "stimmung_label", "thema_label"]
kommentare[spalten].to_csv(ORDNER + "feedback_klassifiziert.csv", index=False)
print("Gespeichert:", ORDNER + "feedback_klassifiziert.csv", f"({len(kommentare)} Zeilen)")
print(kommentare.groupby(["thema", "stimmung"]).size().unstack(fill_value=0))

Gespeichert: project/examples/m04-llm-analysis/l02-feedback/feedback_klassifiziert.csv (50 Zeilen)
stimmung    negativ  neutral  positiv
thema                                
Lieferung         5        5        3
Onlineshop        4        0        4
Preis             2        0        4
Qualität          3        0        5
Service           4        5        2
unklar            2        2        0
